In [ ]:
import torch
import math

regress_save = torch.load("regress_save.pt")
A = regress_save['A']
B = regress_save['B']
u = regress_save['u']

In [ ]:
def sigmoid_normal_large_B_expansion(A, B, u_mean, u_cov, eps=1e-12):
    """
    Approximate

        g = E[(sigmoid(u @ B) * u) @ A.T]

    using the large-|B| expansion

        I_j = E[u_j sigmoid((u @ B)_j)]

    and then

        g = A @ I

    Shapes:
        A:      [128, 513]
        B:      [513, 513]
        u_mean: [513]
        u_cov:  [513, 513]

    Returns:
        g: [128]
        I: [513]
    """

    # Ensure compatible dtype/device
    dtype = A.dtype
    device = A.device
    B = B.to(device=device, dtype=dtype)
    u_mean = u_mean.to(device=device, dtype=dtype)
    u_cov = u_cov.to(device=device, dtype=dtype)

    # Y_j = (u @ B)_j
    #
    # m_j = E[Y_j]
    # q_j^2 = Var[Y_j]
    # c_j = Cov(u_j, Y_j)
    m = u_mean @ B                       # [513]
    var_y = torch.sum(B * (u_cov @ B), dim=0)  # diag(B.T @ Cov @ B), [513]
    q = torch.sqrt(torch.clamp(var_y, min=eps)) # [513]

    # c_j = Cov(u_j, Y_j) = Cov[j, :] @ B[:, j]
    c = torch.sum(u_cov * B.T, dim=1)    # [513]

    # alpha_j = m_j / q_j
    alpha = m / q

    sqrt_2pi = math.sqrt(2.0 * math.pi)
    phi = torch.exp(-0.5 * alpha**2) / sqrt_2pi
    Phi = 0.5 * (1.0 + torch.erf(alpha / math.sqrt(2.0)))

    # Conditional expectation:
    #
    # E[u_j | Y_j = y] = mu_j + beta_j (y - m_j)
    #                  = a_j + beta_j y
    #
    # beta_j = Cov(u_j, Y_j) / Var(Y_j)
    beta = c / torch.clamp(var_y, min=eps)
    a = u_mean - beta * m

    # Leading hard-threshold term:
    #
    # E[u_j 1_{Y_j > 0}]
    # = mu_j Phi(alpha_j) + c_j / q_j * phi(alpha_j)
    I0 = u_mean * Phi + (c / q) * phi

    # Let h_j(y) = E[u_j | Y_j = y] p_Y(y).
    # Expansion:
    #
    # E[u_j sigmoid(Y_j)]
    # = E[u_j 1_{Y_j > 0}]
    #   - pi^2 / 6 h'_j(0)
    #   - 7 pi^4 / 360 h'''_j(0)
    #   + O(q_j^{-6})
    #
    # These are derivatives w.r.t. y.
    p0 = phi / q

    h_prime_0 = p0 * (
        beta + a * m / q**2
    )

    h_third_0 = p0 * (
        a * (-3.0 * m / q**4 + m**3 / q**6)
        + 3.0 * beta * (m**2 / q**4 - 1.0 / q**2)
    )

    I = (
        I0
        - (math.pi**2 / 6.0) * h_prime_0
        - (7.0 * math.pi**4 / 360.0) * h_third_0
    )

    # g = E[(sigmoid(u @ B) * u) @ A.T] = A @ I
    g = A @ I                            # [128]

    return g, I

In [55]:
def zero_inflated_mean_cov(X, unbiased=True, eps=1e-8):
    """
    X: [n_samples, d]

    Returns:
      mean_nonzero: conditional mean E[X_i | X_i != 0], shape [d]
      cov_nonzero: pairwise covariance conditional on both entries nonzero, shape [d, d]

    Missing/insufficient covariance entries are returned as 0, not NaN.
    """
    X = X.float()
    mask = X != 0
    M = mask.float()

    # Conditional mean per coordinate
    counts = M.sum(dim=0)
    safe_counts = counts.clamp_min(1)

    mean_nonzero = (X * M).sum(dim=0) / safe_counts

    # If a coordinate is never nonzero, set its mean to 0
    mean_nonzero = torch.where(
        counts > 0,
        mean_nonzero,
        torch.zeros_like(mean_nonzero)
    )

    # Pairwise counts: number of samples where both i and j are nonzero
    pair_counts = M.T @ M

    # Center only nonzero entries
    X_centered = torch.where(mask, X - mean_nonzero, torch.zeros_like(X))

    cov_num = X_centered.T @ X_centered

    if unbiased:
        denom = pair_counts - 1
        valid = pair_counts >= 2
    else:
        denom = pair_counts
        valid = pair_counts >= 1

    safe_denom = denom.clamp_min(1)
    cov_nonzero = cov_num / safe_denom

    # Replace invalid covariance entries with 0
    cov_nonzero = torch.where(
        valid,
        cov_nonzero,
        torch.zeros_like(cov_nonzero)
    )

    return mean_nonzero, cov_nonzero

In [67]:
# Real data

# mean, cov = zero_inflated_mean_cov(u)
mean = u.mean(dim=0)
cov = torch.cov(u.T)
u_test = u

In [65]:
# True solution
h = (torch.sigmoid(u_test @ B.T) * u_test)
g = h @ A.T

h = h.mean(dim=0)
print(h[:20])

tensor([0.1140, 0.0137, 0.0461, 0.0171, 0.0839, 0.0631, 0.0027, 0.0541, 0.1062,
        0.0097, 0.0155, 0.1087, 0.0348, 0.0025, 0.0000, 0.0347, 0.0492, 0.0216,
        0.0070, 0.0384])


In [68]:
# Approximate solution V1
g_approx, h_approx = sigmoid_normal_large_B_expansion(A, B.T, mean, cov)
print(h_approx[:20])

tensor([0.1258, 0.0285, 0.0515, 0.0223, 0.0845, 0.0632, 0.0171, 0.0401, 0.0887,
        0.0239, 0.0242, 0.1112, 0.0345, 0.0028, 0.0000, 0.0453, 0.0375, 0.0261,
        0.0107, 0.0489])


In [69]:
# Error magnitude:
err = torch.norm(h - h_approx) / torch.norm(h)
print(f'{err = }')
print(f'{torch.norm(h_approx) = }')

err = tensor(0.2458)
torch.norm(h_approx) = tensor(1.1537)


In [ ]:
u.quantile(0.4)
u_nonzero = u[u>=1e-4]
print(u_nonzero.shape, u_nonzero.std())

In [ ]:
from matplotlib import pyplot as plt
# Laplace dist
dist = torch.distributions.Laplace(0, 0.52)
samples = dist.sample([100000])
plt.hist(u_nonzero, bins=50, density=True)
plt.hist(torch.randn(100000).abs() * 0.81, bins=50, density=True)

In [22]:
(u==0).sum() / u.numel()

tensor(0.4402)